# PROC CHART による地域別グリッド負荷と停電のプロファイリング

## エグゼクティブサマリー

あるグリッド運用アナリストが PROC CHART を用いて、4 つのサービス地域と 4 つの発電源にわたるフィーダー回路の計器読み取り値の合成サンプルをプロファイリングします。このノートブックでは、縦棒グラフ、横棒グラフ、ブロックチャート、円グラフを順に取り上げ、発電構成の要約、地域別の平均負荷と総負荷の比較、時間帯別の夕方需要ピークの可視化、発電源別の停電時間のランク付けを行います。これは、エネルギー・公益事業チームがグラフィカルなレポートに着手する前に実施する、迅速なテキスト出力による探索の典型例です。DATA ステップは 1,200 行を要求しますが、このノートブックは非ライセンスモードでレンダリングされており、作業用データセットは最初の 100 件の読み取り値に制限されるため、以下の各チャートはその 100 行のサンプルを要約したものです。

## データソース

| データセット | 行数 | 説明 |
|---------|------|-------------|
| `grid_ops` | 100（合成サンプル） | 地域グリッド上のフィーダー回路計器読み取り値 1 件につき 1 行。`call streaminit(20260531)` と `rand()` を用いてインラインで生成される。DATA ステップのループは 1,200 件の読み取り値を要求するが、非ライセンスモードでは保存されるデータセットが 100 観測に制限されるため、以下のチャートはその 100 行を記述している。 |

# PROC CHART による地域別グリッド負荷と停電のプロファイリング

PROC CHART は、ODS Graphics デバイスを必要とせずに、文字ベースの棒グラフ、ブロックチャート、円グラフをリスティングに直接生成します。そのため、洗練された GCHART や SGPLOT の可視化を作成する前に、負荷と信頼性データの*形状*を*見て*把握したいグリッド運用チームにとって、迅速な初回探索ツールとなります。

このノートブックでは、次のことを行います。

1. 地域グリッド運用者向けに、フィーダー回路の計器読み取り値を合成した 1 か月分を生成する。
2. **発電構成**（発電源別の読み取り値のシェア）をチャート化する。
3. サービス地域間で**平均および総配電負荷**を比較する。
4. 時間帯別に**夕方の需要ピーク**を可視化する。
5. **ブロックチャート**を用いて地域と発電源をクロス集計する。
6. **停電時間**を発電源別にランク付けし、最も信頼性の低いフィードを見つける。

以下のすべてのステートメントとオプションは、標準的な SAS 9.4 PROC CHART 構文です。

## ステップ 1 — 合成運用データの生成

以下の DATA ステップは、1,200 回の反復ループで計器読み取り値を作成します。各読み取り値には、サービス地域、発電源（ガス が支配的で、残りを 風力、太陽光、水力 が占める）、および時間帯が割り当てられます。負荷は 17:00〜21:00 の夕方の時間帯に高くなり（これらの読み取り値をピークとしてフラグ付けします）、停電時間はポアソン分布から抽出します。`call streaminit` が乱数シードを固定するため、データは再現可能です。

ログの NOTE が実際の結果を示しています。この実行は非ライセンスモードのため、保存される `grid_ops` データセットはこれらの読み取り値のうち最初の 100 件に制限されます（100 行、7 列）。以降のすべてのチャートはその 100 行のサンプルを要約しており、各 PROC CHART のログが 100 行を読み取ったことを確認しています。

In [1]:
/* 地域グリッド運用者向けの合成フィーダー回路運用データ */
データ grid_ops;
    呼出 streaminit(20260531);
    長さ region $12 source $12 peak_flag $12;
    見出 region="地域" source="発電源" hour="時間帯"
          load_mwh="配電負荷(MWh)" outage_min="停電時間(分)"
          peak_flag="ピーク区分" meter_id="計器ID";
    配列 regions[4] $12 _temporary_
        ('北部','南部','東部','西部');
    繰返 meter_id = 1 から 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        もし u < 0.42 なら source = 'ガス';
        他 もし u < 0.70 なら source = '風力';
        他 もし u < 0.88 なら source = '太陽光';
        他 source = '水力';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = '北部') + 3 * (region = '西部');
        もし hour >= 17 かつ hour <= 21 なら 繰返;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'ピーク';
        終了;
        他 繰返;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = '通常';
        終了;
        もし load_mwh < 0 なら load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        出力;
    終了;
    削除 r u BASE;
実行;


NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.23 seconds
  cpu   0.23 seconds


## ステップ 2 — 発電構成

各発電源は読み取り値の何パーセントを占めているでしょうか。`TYPE=PERCENT` を指定した縦棒グラフがこれに直接答えます。棒の高さは、全観測のうち各発電源カテゴリに該当する割合（パーセント）です。`source` は文字変数であるため、PROC CHART はその値を自動的に離散カテゴリとして扱います。

In [2]:
処理 chart データ=grid_ops;
    VBAR source / type=PERCENT;
実行;


Percent of 発電源

         |                      *********           
         |                      *********           
   40.00 +                      *********           
         |                      *********           
         |                      *********           
   30.00 +                      *********           
         |           *********  *********           
         |           *********  *********           
   20.00 +           *********  *********           
         |           *********  *********           
         |*********  *********  *********           
         |*********  *********  *********  *********
   10.00 +*********  *********  *********  *********
         |*********  *********  *********  *********
         |*********  *********  *********  *********
         |                                          
         +------------------------------------------+
             水力         風力         ガス         太陽光   
                           発


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## ステップ 3 — 地域別の平均配電負荷

ここでは、件数の集計から測定値の要約へと切り替えます。`SUMVAR=` に `load_mwh` を指定し `TYPE=MEAN` とすることで、棒の長さが地域ごとの平均負荷になります。また、統計量の列を明示的に要求します。`MEAN` は各棒の横に平均値を表示し、`CFREQ` は累積度数の列を追加します。北部 が最も高い平均配電負荷を担っており（平均約 28.0 MWh）、これはデータに組み込まれた地域オフセットと整合します。一方、南部 が最も低くなっています（約 19.8 MWh）。

さらに `ASCENDING` を渡して、棒を平均値の低い順から高い順に並べようとしています。しかし出力では、棒が並べ替えられて**いない**ことに注意してください。棒はカテゴリ順（西部、南部、東部、北部）で表示され、平均値は 24.2、19.8、21.7、28.0 で、短い順から長い順にはなっていません。この PROC CHART 実装では、チャート統計量による棒の並べ替えがまだ実装されていないため、`ASCENDING`/`DESCENDING` は受け付けられるものの現時点では効果がありません。ランキングは、表示された `Mean` 列から読み取ってください。

In [3]:
処理 chart データ=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
実行;


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## ステップ 4 — 地域別の総負荷

ここでは `TYPE=SUM` により、各棒が地域の平均ではなく*総*配電負荷を表すため、高い棒はサンプリングされた読み取り値全体で最も多くの総エネルギーを配電した地域を示します。総配電負荷でも再び 北部 が首位となります。

このステートメントでは `SUBGROUP=peak_flag` も要求しており、読み取り値が夕方のピーク時間帯に該当したかどうかで各棒を分割するよう PROC CHART に指示しています。SAS ではこれにより各棒がサブグループ水準ごとに異なるグリフで区切られ、凡例が表示されます。この実装ではサブグループ分割のレンダリングがまだ行われていないため（文書化された機能ギャップ）、棒はセグメント分割のない `*` の連続として出力されます。棒の*合計*は正しいものの、ピーク時と非ピーク時の内訳はここでは表示されません。ピーク時間帯にどれだけの負荷がかかるかを見るには、ステップ 5 の時間帯別ビューを使用してください。

In [4]:
処理 chart データ=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
実行;


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## ステップ 5 — 1 日を通じた負荷の形状

`hour` は連続変数であるため、4 時間ごとの中心点（2、6、10、14、18、22）で明示的な `MIDPOINTS=` リストを指定して、ビン分割を自分で固定します。各棒は、その時間帯付近の読み取り値の平均配電負荷を示します。18 を中心とする棒が目立つはずです。これはデータに組み込んだ夕方の需要ピークです。

In [5]:
処理 chart データ=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
実行;


Mean of 時間帯

   30.00 +                            *****       
         |                            *****       
         |                            *****       
         |                            *****  *****
         |                            *****  *****
   20.00 +                            *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                          時間帯





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## ステップ 6 — 地域 × 発電源（ブロックチャート）

BLOCK チャートは、小さな二元表をブロックのグリッドとしてレンダリングします。`GROUP=source` と `SUMVAR=load_mwh / TYPE=MEAN` を指定すると、各地域とそれにサービスを供給する発電源をクロス集計し、ブロックの高さが平均負荷に比例します。これは、どの地域と発電源の組み合わせが最も高い平均負荷を担っているかを見つけるためのコンパクトな方法です。

In [6]:
処理 chart データ=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
実行;


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## ステップ 7 — 円グラフとしての発電構成

ステップ 2 と同じ発電源シェアの情報を、円グラフとして描画します。`TYPE=PERCENT` を指定した PIE は、各スライスを総読み取り値に対する割合（パーセント）でサイズ調整し、図の横にスライスのパーセントの凡例を表示します。

In [7]:
処理 chart データ=grid_ops;
    PIE source / type=PERCENT;
実行;


Pie chart of 発電源

             発電源     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
              水力       14.00     14.0%  ####
              風力       28.00     28.0%  ********
              ガス       45.00     45.0%  ++++++++++++++
             太陽光       13.00     13.0%  OOOO





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## ステップ 8 — 発電源別の停電時間

最後に信頼性をランク付けします。`SUMVAR=outage_min` と `TYPE=SUM` により、発電源ごとの停電時間を合計します。最も成績の悪い発電源を上位に浮上させようと `DESCENDING` を渡していますが、ステップ 3 と同様に棒は並べ替えられません。棒はカテゴリ順（水力、風力、ガス、太陽光）で表示され、棒の並べ替えはまだ実装されていません。ランキングは、表示された `Sum` 列から読み取ってください。このサンプルでは ガス が総停電時間の最も多くを占めており（122）、風力（64）、太陽光（43）、水力（38）を大きく上回っています。これは発電構成を反映しています。ガス は最も多くの読み取り値にサービスを供給しているため、全体として最も多くの停電時間を累積します。

In [8]:
処理 chart データ=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DESCENDING;
実行;


Sum of 発電源

      発電源                                                   Sum        Cum.     Percent
                                                                        Sum            
       水力  ************                                      38          38       14.00
       風力  *********************                             64         102       28.00
       ガス  ****************************************         122         224       45.00
      太陽光  **************                                    43         267       13.00





NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 結果の解釈

これらのチャートを合わせて読むことで、運用チームは迅速に状況を把握できます（この実行で捕捉された 100 行のサンプルについて）。

- **発電構成（ステップ 2 および 7）。** ガス が最も大きな読み取り値のシェア（約 45%）を占め、風力 が 2 位（約 28%）、水力 と 太陽光 がそれに続きます（約 14% と 13%）。縦棒グラフと円グラフは同じ内容を 2 通りの方法で示しており、有用な整合性チェックになります。
- **地域別の負荷（ステップ 3 および 4）。** 平均負荷の HBAR は、北部 が最も高い平均配電負荷（平均約 28 MWh）で、南部 が最も低い（約 20 MWh）ことを示しており、データに組み込まれた地域オフセットと整合します。SUM チャートは、北部 が総エネルギーでも最多であることを確認しています。
- **日中の負荷形状（ステップ 5）。** 18 時を中心とする中心点の棒が明らかに最も高く、データに組み込んだ 17:00〜21:00 の需要ピークを裏付けています。まさに公益事業者がデマンドレスポンスと容量計画に注力する箇所です。
- **信頼性（ステップ 8）。** 発電源別に停電時間を合計すると、このサンプルでは ガス がダウンタイムの最大の要因（122 分）として浮かび上がり、メンテナンスレビューの当然の次なる対象となります。ただしこれは主に、ガス が最も多くの読み取り値にサービスを供給していることを反映しています。

ここで使用した 2 つの表示オプション、すなわち `ASCENDING`/`DESCENDING` による棒の並べ替え（ステップ 3 および 8）と `SUBGROUP=` による棒の分割（ステップ 4）は、パーサーには受け付けられるものの、この PROC CHART 実装ではまだレンダリングされません。そのため、ランキングや内訳は棒の順序や陰影ではなく、表示された統計量の列から読み取ります。

PROC CHART は文字出力専用であるため、役員会向けの見栄えのする可視化には、チームはこれらと同じビューを PROC GCHART や PROC SGPLOT に移すことになります。しかし、新しい抽出データに対するゼロセットアップの初回パスとして、これらのチャートは運用上の疑問、すなわち構成、規模、タイミングに数秒で答えます。